In [1]:
pip install xgboost optuna

In [2]:
pip install optuna-dashboard

In [20]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import mean_squared_error, r2_score
import xgboost as xgb
import kagglehub
from kagglehub import KaggleDatasetAdapter
import optuna
from optuna.visualization import plot_optimization_history, plot_param_importances, plot_parallel_coordinate
import threading
import time
from wsgiref.simple_server import make_server
from optuna_dashboard import wsgi
from google.colab import output

In [4]:
file_path = "diamonds.csv"

df = kagglehub.load_dataset(
  KaggleDatasetAdapter.PANDAS,
  "resulcaliskan/diamonds",
  file_path
)

print("First 5 records:", df.head())

/tmp/ipykernel_47631/2448166788.py:3: DeprecationWarning: Use dataset_load() instead of load_dataset(). load_dataset() will be removed in a future version.
  df = kagglehub.load_dataset(


Using Colab cache for faster access to the 'diamonds' dataset.
First 5 records:    carat      cut color clarity  depth  table     x     y     z  price
0   0.23    Ideal     E     SI2   61.5   55.0  3.95  3.98  2.43    326
1   0.21  Premium     E     SI1   59.8   61.0  3.89  3.84  2.31    326
2   0.23     Good     E     VS1   56.9   65.0  4.05  4.07  2.31    327
3   0.29  Premium     I     VS2   62.4   58.0  4.20  4.23  2.63    334
4   0.31     Good     J     SI2   63.3   58.0  4.34  4.35  2.75    335


In [5]:
categorical_cols = ['cut', 'color', 'clarity']
df = pd.get_dummies(df, columns=categorical_cols, drop_first=True)

X = df.drop('price', axis=1)
y = df['price']

In [6]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [22]:
param_grid = {
    'n_estimators': [100, 200, 300],
    'max_depth': [3, 6, 9],
    'learning_rate': [0.01, 0.1, 0.3],
}

model = xgb.XGBRegressor(random_state=42, n_jobs=-1)

grid_search = GridSearchCV(
    model,
    param_grid,
    cv=3,
    scoring='neg_mean_squared_error'
)

grid_search.fit(X_train, y_train)

127.0.0.1 - - [23/Aug/2026 21:59:37] "GET /api/studies/1?after=21 HTTP/1.1" 200 2156
127.0.0.1 - - [23/Aug/2026 21:59:47] "GET /api/studies/1?after=21 HTTP/1.1" 200 2156
127.0.0.1 - - [23/Aug/2026 21:59:58] "GET /api/studies/1?after=21 HTTP/1.1" 200 2156
127.0.0.1 - - [23/Aug/2026 22:00:08] "GET /api/studies/1?after=21 HTTP/1.1" 200 2156
127.0.0.1 - - [23/Aug/2026 22:00:18] "GET /api/studies/1?after=21 HTTP/1.1" 200 2156
127.0.0.1 - - [23/Aug/2026 22:00:29] "GET /api/studies/1?after=21 HTTP/1.1" 200 2156
127.0.0.1 - - [23/Aug/2026 22:00:39] "GET /api/studies/1?after=21 HTTP/1.1" 200 2156
127.0.0.1 - - [23/Aug/2026 22:00:50] "GET /api/studies/1?after=21 HTTP/1.1" 200 2156
127.0.0.1 - - [23/Aug/2026 22:01:01] "GET /api/studies/1?after=21 HTTP/1.1" 200 2156
127.0.0.1 - - [23/Aug/2026 22:01:12] "GET /api/studies/1?after=21 HTTP/1.1" 200 2156
127.0.0.1 - - [23/Aug/2026 22:01:23] "GET /api/studies/1?after=21 HTTP/1.1" 200 2156
127.0.0.1 - - [23/Aug/2026 22:01:34] "GET /api/studies/1?after=21

GridSearchCV(cv=3,
             estimator=XGBRegressor(base_score=None, booster=None,
                                    callbacks=None, colsample_bylevel=None,
                                    colsample_bynode=None,
                                    colsample_bytree=None, device=None,
                                    early_stopping_rounds=None,
                                    enable_categorical=True, eval_metric=None,
                                    feature_types=None, feature_weights=None,
                                    gamma=None, grow_policy=None,
                                    importance_type=None,
                                    interaction_constraints=None,...
                                    max_cat_threshold=None,
                                    max_cat_to_onehot=None, max_delta_step=None,
                                    max_depth=None, max_leaves=None,
                                    min_child_weight=None, missing=nan,
                                    monotone_constraints=None,
                                    multi_strategy=None, n_estimators=None,
                                    n_jobs=-1, num_parallel_tree=None, ...),
             param_grid={'learning_rate': [0.01, 0.1, 0.3],
                         'max_depth': [3, 6, 9],
                         'n_estimators': [100, 200, 300]},
             scoring='neg_mean_squared_error')

In [23]:
print("Лучшие параметры:", grid_search.best_params_)

best_model = grid_search.best_estimator_
y_pred = best_model.predict(X_test)

rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)

print(f"RMSE: {rmse:.2f}")
print(f"r2: {r2:.4f}")

Лучшие параметры: {'learning_rate': 0.1, 'max_depth': 9, 'n_estimators': 200}
RMSE: 567.65
r2: 0.9797


In [12]:
def objective(trial):
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 100, 300),
        'max_depth': trial.suggest_int('max_depth', 3, 9),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3),
    }
    model = xgb.XGBRegressor(**params, random_state=42)
    model.fit(X_train, y_train)
    return -model.score(X_test, y_test)

In [13]:
study = optuna.create_study()
study.optimize(objective, n_trials=10)

[I 2026-08-23 21:34:53,925] A new study created in memory with name: no-name-1b8e31b9-e3d3-4ce1-bdbf-b480a186a123
[I 2026-08-23 21:34:56,421] Trial 0 finished with value: -0.9734269976615906 and parameters: {'n_estimators': 242, 'max_depth': 8, 'learning_rate': 0.017823322030065335}. Best is trial 0 with value: -0.9734269976615906.
[I 2026-08-23 21:34:56,945] Trial 1 finished with value: -0.9750890731811523 and parameters: {'n_estimators': 105, 'max_depth': 5, 'learning_rate': 0.17508146426964527}. Best is trial 1 with value: -0.9750890731811523.
[I 2026-08-23 21:34:57,767] Trial 2 finished with value: -0.9759179949760437 and parameters: {'n_estimators': 211, 'max_depth': 4, 'learning_rate': 0.19277582368465299}. Best is trial 2 with value: -0.9759179949760437.
[I 2026-08-23 21:34:58,479] Trial 3 finished with value: -0.9636420607566833 and parameters: {'n_estimators': 151, 'max_depth': 5, 'learning_rate': 0.04013144522751006}. Best is trial 2 with value: -0.9759179949760437.
[I 2026-0

In [14]:
print("Лучшие параметры:", study.best_params)
print(f"r2: {-study.best_value:.4f}")

Лучшие параметры: {'n_estimators': 266, 'max_depth': 7, 'learning_rate': 0.09025371703610073}
r2: 0.9802


In [15]:
port = 8081
storage = optuna.storages.RDBStorage("sqlite:///diamonds_study.db")
study = optuna.create_study(
    direction='minimize',
    storage=storage,
    study_name="diamonds_optimization",
    load_if_exists=True
)

app = wsgi(storage)
httpd = make_server("localhost", port, app)
thread = threading.Thread(target=httpd.serve_forever)
thread.daemon = True
thread.start()
time.sleep(2)

output.serve_kernel_port_as_iframe(port, path='/dashboard/')

[I 2026-08-23 21:35:08,092] Using an existing study with name 'diamonds_optimization' instead of creating a new one.


<IPython.core.display.Javascript object>

In [16]:
fig1 = plot_optimization_history(study)
fig1.show()

127.0.0.1 - - [23/Aug/2026 21:35:12] "GET /dashboard/ HTTP/1.1" 200 4145


In [17]:
fig2 = plot_param_importances(study)
fig2.show()

127.0.0.1 - - [23/Aug/2026 21:35:12] "GET /static/bundle.js HTTP/1.1" 200 4159096


In [18]:
print("Важность признаков")

feature_importance = pd.DataFrame({
    'Признак': X.columns,
    'Важность': best_model.feature_importances_
}).sort_values('Важность', ascending=False)

print(feature_importance)

Важность признаков
          Признак  Важность
0           carat      1332
4               y      1304
5               z      1138
3               x      1018
1           depth       955
2           table       513
18    clarity_SI2       247
14        color_I       233
15        color_J       224
13        color_H       223
17    clarity_SI1       181
12        color_G       173
20    clarity_VS2       170
22   clarity_VVS2       169
21   clarity_VVS1       161
16     clarity_IF       157
19    clarity_VS1       153
7       cut_Ideal       115
8     cut_Premium       106
11        color_F       101
10        color_E        98
9   cut_Very Good        75
6        cut_Good        24
